# Defect-size sweep on Google Colab

**Upload to Google Drive, into one folder (default name `defect_size_sweep`):**

1. the `src` folder from the project (it must contain `run_defect_size_sweep.py` and `evaluation.py`)
2. the MVTec texture data, laid out exactly like on the laptop:
   `data/mvtec/<texture>/train/good/*.png` and `data/mvtec/<texture>/test/good/*.png`
   for carpet, grid, leather, tile and wood.

Nothing else is needed (no ground-truth masks, no results).

**Important:** upload the laptop's own `data/mvtec` folder, not a fresh MVTec download.
The laptop has only 30 training images per texture, and Route B draws its 4 references
from those by index. A full MVTec download (hundreds of training images) would give
different references and therefore different numbers.

Expected Drive layout:
```
MyDrive/defect_size_sweep/
    src/run_defect_size_sweep.py
    src/evaluation.py
    data/mvtec/carpet/train/good/*.png
    data/mvtec/carpet/test/good/*.png
    ... (grid, leather, tile, wood)
```

Use a GPU runtime: *Runtime → Change runtime type → GPU*.
Every finished (texture, size, method, seed) result is written straight to Drive,
so if the session disconnects, run all cells again and the sweep continues where it stopped.

In [ ]:
# 1) Which GPU did Colab give us?
!nvidia-smi

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Paths (change PROJECT if you used another folder name)
import os
PROJECT = '/content/drive/MyDrive/defect_size_sweep'
DATA    = f'{PROJECT}/data/mvtec'
RESULTS = f'{PROJECT}/results/defect_size_sweep'
SMOKE   = f'{PROJECT}/results/defect_size_sweep_quick'

for p in [f'{PROJECT}/src/run_defect_size_sweep.py', f'{PROJECT}/src/evaluation.py'] + \
         [f'{DATA}/{t}/{s}/good' for t in ['carpet', 'grid', 'leather', 'tile', 'wood'] for s in ['train', 'test']]:
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)
for t in ['carpet', 'grid', 'leather', 'tile', 'wood']:
    d = f'{DATA}/{t}/train/good'
    if os.path.isdir(d):
        print(t, 'train images:', len([f for f in os.listdir(d) if f.endswith('.png')]), '(laptop: 30)')
os.chdir(PROJECT)

In [ ]:
# 4) Install only what is missing (torch, torchvision, numpy, scipy, sklearn and matplotlib are normally preinstalled)
import importlib, subprocess, sys
need = {'cv2': 'opencv-python-headless', 'sklearn': 'scikit-learn', 'scipy': 'scipy',
        'matplotlib': 'matplotlib', 'PIL': 'pillow', 'torch': 'torch', 'torchvision': 'torchvision'}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
print('Installed:' if missing else 'Nothing missing.', ' '.join(missing))

In [ ]:
# 5a) Smoke test first: compare these numbers with the laptop's smoke test before trusting Colab.
!python src/run_defect_size_sweep.py --quick --textures carpet --sizes 1 8 --seeds 42 \
    --data-root "$DATA" --results-dir "$SMOKE"

In [ ]:
# 5b) Full sweep (5 textures x 8 sizes x (Route A + 10 Route B seeds)).
#     Results go to Drive after every step; if Colab disconnects, re-run cells 2, 3 and this one.
!python src/run_defect_size_sweep.py --data-root "$DATA" --results-dir "$RESULTS"

In [ ]:
# 6) Final figure and summary
from IPython.display import Image, display
display(Image(filename=f'{RESULTS}/defect_size_sweep.png'))
print(open(f'{RESULTS}/summary.txt', encoding='utf-8').read())

In [ ]:
# Optional: make sure everything is written to Drive before you close the session
drive.flush_and_unmount()